# Задача 9. Hand-crafted graph features

В этой работе решается бинарная классификация графов с помощью hand-crafted graph kernels:

- shortest path kernel из количества кратчайших путей разной длины;
- Weisfeiler-Lehman subtree kernel;
- SVC с `kernel="precomputed"`, подбором `C` и метриками качества.

In [ ]:
# mypy: ignore-errors

import random
from collections import Counter
from collections.abc import Hashable
from dataclasses import dataclass

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.svm import SVC

SEED = 42
N_GRAPHS_PER_CLASS = 120
TEST_SIZE = 0.25

random.seed(SEED)
np.random.seed(SEED)

## 1. Датасет для бинарной классификации графов

Сгенерируем сбалансированный датасет из двух типов связных графов:

- класс `0`: случайные деревья с небольшим числом дополнительных рёбер;
- класс `1`: connected Watts-Strogatz графы с циклами и локальной кластеризацией.

Размеры графов берутся из одного диапазона для обоих классов, поэтому задача не сводится только к числу вершин.

In [ ]:
def add_degree_labels(graph: nx.Graph) -> nx.Graph:
    graph = nx.convert_node_labels_to_integers(graph)
    degree_labels = {node: str(degree) for node, degree in graph.degree()}
    nx.set_node_attributes(graph, degree_labels, "label")
    return graph


def make_tree_like_graph(n_nodes: int, seed: int) -> nx.Graph:
    rng = np.random.default_rng(seed)
    prufer_sequence = rng.integers(0, n_nodes, size=n_nodes - 2).tolist()
    graph = nx.from_prufer_sequence(prufer_sequence)
    possible_edges = [
        (u, v)
        for u in range(n_nodes)
        for v in range(u + 1, n_nodes)
        if not graph.has_edge(u, v)
    ]
    n_extra_edges = int(rng.integers(0, 3))
    if n_extra_edges > 0:
        edge_indices = rng.choice(
            len(possible_edges),
            size=n_extra_edges,
            replace=False,
        )
        graph.add_edges_from(possible_edges[idx] for idx in edge_indices)
    return add_degree_labels(graph)


def make_small_world_graph(n_nodes: int, seed: int) -> nx.Graph:
    k_neighbors = 4 if n_nodes > 8 else 2
    graph = nx.connected_watts_strogatz_graph(
        n=n_nodes,
        k=k_neighbors,
        p=0.2,
        tries=100,
        seed=seed,
    )
    return add_degree_labels(graph)


def generate_graph_dataset(
    n_graphs_per_class: int,
    seed: int = SEED,
) -> tuple[list[nx.Graph], np.ndarray]:
    rng = np.random.default_rng(seed)
    graphs = []
    labels = []

    for class_label in [0, 1]:
        for graph_idx in range(n_graphs_per_class):
            n_nodes = int(rng.integers(18, 46))
            graph_seed = seed + class_label * 10_000 + graph_idx
            if class_label == 0:
                graph = make_tree_like_graph(n_nodes, graph_seed)
            else:
                graph = make_small_world_graph(n_nodes, graph_seed)
            graphs.append(graph)
            labels.append(class_label)

    labels_array = np.asarray(labels, dtype=int)
    permutation = rng.permutation(len(graphs))
    shuffled_graphs = [graphs[idx] for idx in permutation]
    return shuffled_graphs, labels_array[permutation]


graphs, y = generate_graph_dataset(N_GRAPHS_PER_CLASS)
print(f"n_graphs={len(graphs)}, class_counts={Counter(y)}")

In [ ]:
def graph_summary(graph: nx.Graph, label: int) -> dict[str, float | int]:
    return {
        "label": label,
        "n_nodes": graph.number_of_nodes(),
        "n_edges": graph.number_of_edges(),
        "avg_clustering": nx.average_clustering(graph),
        "diameter": nx.diameter(graph),
        "avg_shortest_path": nx.average_shortest_path_length(graph),
    }


summary_df = pd.DataFrame(
    graph_summary(graph, int(label)) for graph, label in zip(graphs, y, strict=True)
)
display(summary_df.groupby("label").mean().round(3))

In [ ]:
train_graphs, test_graphs, y_train, y_test = train_test_split(
    graphs,
    y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

print(
    "split:",
    {
        "train": len(train_graphs),
        "test": len(test_graphs),
        "train_classes": Counter(y_train),
        "test_classes": Counter(y_test),
    },
)

## 2. Shortest path kernel

Для каждого графа считаем вектор `phi(G)`, где координата `d` — количество неупорядоченных пар вершин с кратчайшим путём длины `d`. Kernel между двумя графами — `phi(G_1) @ phi(G_2)`.

In [ ]:
def shortest_path_feature_counter(graph: nx.Graph) -> Counter[int]:
    path_lengths = dict(nx.shortest_path_length(graph))
    features: Counter[int] = Counter()

    for source, target_lengths in path_lengths.items():
        for target, distance in target_lengths.items():
            if source < target:
                features[int(distance)] += 1

    return features


def counters_to_matrix(
    counters: list[Counter[Hashable]],
    vocabulary: list[Hashable],
) -> np.ndarray:
    matrix = np.zeros((len(counters), len(vocabulary)), dtype=float)
    column_by_distance = {distance: idx for idx, distance in enumerate(vocabulary)}

    for row_idx, counter in enumerate(counters):
        for distance, count in counter.items():
            if distance in column_by_distance:
                matrix[row_idx, column_by_distance[distance]] = count

    return matrix


def shortest_path_kernel(
    train_graphs: list[nx.Graph],
    test_graphs: list[nx.Graph],
) -> tuple[np.ndarray, np.ndarray]:
    train_counters = [shortest_path_feature_counter(graph) for graph in train_graphs]
    test_counters = [shortest_path_feature_counter(graph) for graph in test_graphs]
    distances = sorted(
        {distance for counter in train_counters + test_counters for distance in counter}
    )

    x_train = counters_to_matrix(train_counters, distances)
    x_test = counters_to_matrix(test_counters, distances)
    k_train = x_train @ x_train.T
    k_test = x_test @ x_train.T
    return k_train, k_test


K_train_sp, K_test_sp = shortest_path_kernel(train_graphs, test_graphs)
print(K_train_sp.shape, K_test_sp.shape)

In [ ]:
@dataclass
class KernelExperimentResult:
    name: str
    best_params: dict[str, float]
    metrics: dict[str, float]
    estimator: SVC
    y_pred: np.ndarray
    y_score: np.ndarray


def train_precomputed_svc(
    kernel_name: str,
    k_train: np.ndarray,
    k_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
) -> KernelExperimentResult:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    grid = GridSearchCV(
        estimator=SVC(kernel="precomputed"),
        param_grid={"C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=-1,
    )
    grid.fit(k_train, y_train)

    estimator = grid.best_estimator_
    y_pred = estimator.predict(k_test)
    y_score = estimator.decision_function(k_test)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score),
        "best_cv_balanced_accuracy": grid.best_score_,
    }
    return KernelExperimentResult(
        name=kernel_name,
        best_params=grid.best_params_,
        metrics=metrics,
        estimator=estimator,
        y_pred=y_pred,
        y_score=y_score,
    )


def show_experiment_result(result: KernelExperimentResult) -> None:
    print(result.name)
    print("best params:", result.best_params)
    display(pd.DataFrame([result.metrics]).round(4))
    display(
        pd.DataFrame(
            confusion_matrix(y_test, result.y_pred),
            index=["true 0", "true 1"],
            columns=["pred 0", "pred 1"],
        )
    )
    print(classification_report(y_test, result.y_pred, digits=4))

## 3. SVC со shortest path kernel

`SVC(kernel="precomputed")` получает готовые kernel matrices. Гиперпараметр `C` выбирается по 5-fold CV на train.

In [ ]:
sp_result = train_precomputed_svc(
    "Shortest path kernel",
    K_train_sp,
    K_test_sp,
    y_train,
    y_test,
)
show_experiment_result(sp_result)

## 4. Weisfeiler-Lehman subtree kernel

WL kernel итеративно уточняет метки вершин: новая метка вершины зависит от текущей метки и отсортированного набора меток соседей. Вектор признаков графа строится как суммарная гистограмма всех WL-меток по итерациям.

In [ ]:
def initial_node_labels(graph: nx.Graph) -> dict[int, str]:
    labels = nx.get_node_attributes(graph, "label")
    if labels:
        return {int(node): str(label) for node, label in labels.items()}
    return {int(node): str(degree) for node, degree in graph.degree()}


def node_signature(
    graph: nx.Graph,
    labels: dict[int, str],
    node: int,
) -> tuple[str, tuple[str, ...]]:
    neighbor_labels = sorted(
        labels[int(neighbor)] for neighbor in graph.neighbors(node)
    )
    return labels[int(node)], tuple(neighbor_labels)


def wl_feature_counters(
    graphs: list[nx.Graph],
    n_iterations: int = 4,
) -> list[Counter[str]]:
    current_labels = [initial_node_labels(graph) for graph in graphs]
    graph_features = [Counter(labels.values()) for labels in current_labels]

    for iteration in range(n_iterations):
        signatures_by_graph = []
        all_signatures = []

        for graph, labels in zip(graphs, current_labels, strict=True):
            graph_signatures = {
                int(node): node_signature(graph, labels, int(node))
                for node in graph.nodes()
            }
            signatures_by_graph.append(graph_signatures)
            all_signatures.extend(graph_signatures.values())

        compressed_labels = {
            signature: f"wl_{iteration}_{idx}"
            for idx, signature in enumerate(sorted(set(all_signatures)))
        }
        current_labels = [
            {
                node: compressed_labels[signature]
                for node, signature in graph_signatures.items()
            }
            for graph_signatures in signatures_by_graph
        ]

        for features, labels in zip(
            graph_features,
            current_labels,
            strict=True,
        ):
            features.update(labels.values())

    return graph_features


def weisfeiler_lehman_kernel(
    train_graphs: list[nx.Graph],
    test_graphs: list[nx.Graph],
    n_iterations: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    all_graphs = train_graphs + test_graphs
    counters = wl_feature_counters(all_graphs, n_iterations=n_iterations)
    vocabulary = sorted({label for counter in counters for label in counter})
    feature_matrix = counters_to_matrix(counters, vocabulary)

    x_train = feature_matrix[: len(train_graphs)]
    x_test = feature_matrix[len(train_graphs) :]
    k_train = x_train @ x_train.T
    k_test = x_test @ x_train.T
    return k_train, k_test


K_train_wl, K_test_wl = weisfeiler_lehman_kernel(
    train_graphs,
    test_graphs,
    n_iterations=4,
)
print(K_train_wl.shape, K_test_wl.shape)

In [ ]:
wl_result = train_precomputed_svc(
    "Weisfeiler-Lehman kernel",
    K_train_wl,
    K_test_wl,
    y_train,
    y_test,
)
show_experiment_result(wl_result)

## 5. Сравнение kernels и выводы

Сравним kernels по test-метрикам. Для бинарной сбалансированной задачи важны `balanced_accuracy`, `f1` и `roc_auc`; accuracy оставлена для полноты.

In [ ]:
results = [sp_result, wl_result]
comparison_rows = []
for result in results:
    comparison_rows.append(
        {
            "kernel": result.name,
            "best_C": result.best_params["C"],
            **result.metrics,
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "balanced_accuracy",
    ascending=False,
)
display(comparison_df.round(4))

In [ ]:
def plot_kernel_matrix(kernel: np.ndarray, title: str) -> None:
    plt.figure(figsize=(5, 4))
    plt.imshow(kernel, cmap="viridis")
    plt.colorbar(label="kernel value")
    plt.title(title)
    plt.xlabel("train graph index")
    plt.ylabel("train graph index")
    plt.tight_layout()
    plt.show()


plot_kernel_matrix(K_train_sp, "Shortest path train kernel")
plot_kernel_matrix(K_train_wl, "Weisfeiler-Lehman train kernel")

In [ ]:
best_result = comparison_df.iloc[0]
sp_balanced_accuracy = sp_result.metrics["balanced_accuracy"]
wl_balanced_accuracy = wl_result.metrics["balanced_accuracy"]
wl_delta = wl_balanced_accuracy - sp_balanced_accuracy

print("Выводы")
print("======")
print(
    "1. Shortest path kernel описывает глобальную форму графа через "
    "распределение расстояний между вершинами. Для сгенерированных "
    "классов это сильный признак: деревья имеют более длинные пути, "
    "а small-world графы короче и плотнее."
)
print(
    "2. Weisfeiler-Lehman kernel использует локальные окрестности и "
    "начальные degree labels. Он лучше ловит различия в локальной "
    "структуре, циклах и степенях вершин."
)
print(
    f"3. WL kernel изменил balanced accuracy на {wl_delta:+.4f} "
    "относительно shortest path kernel."
)
print(
    "4. Лучший kernel в этом запуске: "
    f"{best_result['kernel']} с balanced_accuracy="
    f"{best_result['balanced_accuracy']:.4f}."
)